---
# <div align="center"><font color='red'>  </font></div>
# <div align="center"><font color='red'> COSC 2779/2972 | Deep Learning  </font></div>
## <div align="center"> <font color='red'> Week 2 Lectorial Example: **Feed-Forward Neural Networks**</font></div>
---


## Description of the dataset

This UCI dataset contains information on default payments, demographic factors, credit data, history of payment, and bill statements of credit card clients in Taiwan from April 2005 to September 2005. The dataset can be found at [UCI ML Repo](https://archive.ics.uci.edu/static/public/350/default+of+credit+card+clients.zip).

The dataset is composed of 24 variables in total. The first variables contain information about the user personal information:

- ID: ID of each client, categorical variable
- LIMIT_BAL: Amount of given credit in New Taiwan dollars (includes individual and family/supplementary credit)
- SEX: Gender, categorical variable (1=male, 2=female)
- EDUCATION: level of education, categorical variable (1=graduate school, 2=university, 3=high school, 4=others)
- MARRIAGE: Marital status, categorical variable (1=married, 2=single, 3=others)
- AGE: Age in years, numerical variable


Others variables contains information about the history of past payments, the following attributes track the past monthly payment records, i.e. the delay of the payment referred to a specific month:

- PAY_0: Repayment status in September 2005 (-1=pay duly, 1=payment delay for one month, 2=payment delay for two months, … 8=payment delay for eight months, 9=payment delay for nine months and above)
- PAY_2: Repayment status in August 2005 (-1=pay duly, 1=payment delay for one month, 2=payment delay for two months, … 8=payment delay for eight months, 9=payment delay for nine months and above)
- PAY_3: Repayment status in July 2005 (-1=pay duly, 1=payment delay for one month, 2=payment delay for two months, … 8=payment delay for eight months, 9=payment delay for nine months and above)
- PAY_4: Repayment status in June 2005 (-1=pay duly, 1=payment delay for one month, 2=payment delay for two months, … 8=payment delay for eight months, 9=payment delay for nine months and above)
- PAY_5: Repayment status in May 2005 (-1=pay duly, 1=payment delay for one month, 2=payment delay for two months, … 8=payment delay for eight months, 9=payment delay for nine months and above)
- PAY_6: Repayment status in April 2005 (-1=pay duly, 1=payment delay for one month, 2=payment delay for two months, … 8=payment delay for eight months, 9=payment delay for nine months and above)

The following attributes instead consider the information related to the amount of bill statement, i.e. a monthly report that credit card companies issue to credit card holders in a specific month:

- BILL_AMT1: Amount of bill statement in September, 2005 (New Taiwan dollar)
- BILL_AMT2: Amount of bill statement in August, 2005 (New Taiwan dollar)
- BILL_AMT3: Amount of bill statement in July, 2005 (New Taiwan dollar)
- BILL_AMT4: Amount of bill statement in June, 2005 (New Taiwan dollar)
- BILL_AMT5: Amount of bill statement in May, 2005 (New Taiwan dollar)
- BILL_AMT6: Amount of bill statement in April, 2005 (New Taiwan dollar)

The last variables instead consider the amount of previous payment in a specific month:

- PAY_AMT1: Amount of previous payment in September, 2005 (New Taiwan dollar)
- PAY_AMT2: Amount of previous payment in August, 2005 (New Taiwan dollar)
- PAY_AMT3: Amount of previous payment in July, 2005 (New Taiwan dollar)
- PAY_AMT4: Amount of previous payment in June, 2005 (New Taiwan dollar)
- PAY_AMT5: Amount of previous payment in May, 2005 (New Taiwan dollar)
- PAY_AMT6: Amount of previous payment in April, 2005 (New Taiwan dollar)

The variable to predict is given by:

- default payment next month: indicate whether the credit card holders are defaulters or non-defaulters (1=yes, 0=no)

## Setting up the Notebook

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

import numpy as np
import pandas as pd

from matplotlib import pyplot as plt
from sklearn.metrics import accuracy_score, precision_score, recall_score, classification_report

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

### Training Utilities
A `plotter` function to visualise training and validation curves.

In [ ]:
from itertools import cycle

m_histories = {}

def plotter(history_hold, metric='accuracy', ylim=[0.0, 1.0]):
    """Plot training curves for one or more models."""
    cycol = cycle('bgrcmk')
    plt.figure(figsize=(10, 5))
    for name, hist in history_hold.items():
        c = next(cycol)
        epochs = range(1, len(hist[metric]) + 1)
        plt.plot(epochs, hist[metric], c + '-', label=name + '_train')
        plt.plot(epochs, hist['val_' + metric], c + '--', label=name + '_val')
    plt.legend()
    plt.ylim(ylim)
    plt.xlabel('Epoch')
    plt.ylabel(metric.capitalize())
    plt.grid(True)
    plt.title(f'Training History — {metric.capitalize()}')
    plt.show()

## Load the Dataset

Let's load the dataset and set it up for use with deep learning models.

The dataset can be downloaded from the [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/350/default+of+credit+card+clients).

In [ ]:
import os
import zipfile
import urllib.request

DATA_FILE = 'default_of_credit_card_clients.csv'

if not os.path.exists(DATA_FILE):
    # Option 1 (default): Download directly from the UCI repository
    ZIP_URL = 'https://archive.ics.uci.edu/static/public/350/default+of+credit+card+clients.zip'
    ZIP_FILE = 'default_of_credit_card_clients.zip'

    print('Downloading dataset from UCI repository...')
    urllib.request.urlretrieve(ZIP_URL, ZIP_FILE)

    with zipfile.ZipFile(ZIP_FILE, 'r') as z:
        z.extractall('.')
        print(f'Extracted: {z.namelist()}')

    # The zip contains an .xls file — convert it to CSV
    # (the xls file has a header row at row 0 that needs to be skipped)
    XLS_FILE = 'default of credit card clients.xls'
    if os.path.exists(XLS_FILE):
        data = pd.read_excel(XLS_FILE, header=1)
        data.to_csv(DATA_FILE, index=False)
        print(f'Saved as {DATA_FILE}')

    # --- Alternative options (uncomment if the above doesn't work) ---

    # Option 2: Upload the CSV manually via Colab's file picker
    # from google.colab import files
    # uploaded = files.upload()  # select the CSV file when prompted

    # Option 3: Mount Google Drive (if you've stored the file there)
    # from google.colab import drive
    # drive.mount('/content/drive')
    # !cp /content/drive/'My Drive'/COSC2779/Lectures/Week02/default_of_credit_card_clients.csv .

data = pd.read_csv(DATA_FILE)
data = data.drop('ID', axis=1)
print(f'Dataset shape: {data.shape}')
data.head()

## Data Exploration

### Data Distribution

In [ ]:
# Identify column types
categorical_cols = ['SEX', 'EDUCATION', 'MARRIAGE'] + ['PAY_' + str(i) for i in [0, 2, 3, 4, 5, 6]]
target_col = 'default payment next month'
numerical_cols = [c for c in data.columns if c not in categorical_cols + [target_col]]

# --- Numerical features ---
n = len(numerical_cols)
ncols_plot = 4
nrows_plot = int(np.ceil(n / ncols_plot))

fig, axes = plt.subplots(nrows_plot, ncols_plot, figsize=(18, 3.5 * nrows_plot))
fig.suptitle('Distribution of Numerical Features', fontsize=14, y=1.01)
for i, col in enumerate(numerical_cols):
    ax = axes.flat[i]
    ax.hist(data[col], bins=30, alpha=0.5, color='steelblue', edgecolor='white', density=True)
    ax.set_title(col, fontsize=10)
    ax.tick_params(labelsize=8)
for j in range(n, axes.size):
    axes.flat[j].set_visible(False)
plt.tight_layout()
plt.show()

# --- Categorical features ---
n = len(categorical_cols)
ncols_plot = 4
nrows_plot = int(np.ceil(n / ncols_plot))

fig, axes = plt.subplots(nrows_plot, ncols_plot, figsize=(18, 3.5 * nrows_plot))
fig.suptitle('Distribution of Categorical Features', fontsize=14, y=1.01)
for i, col in enumerate(categorical_cols):
    ax = axes.flat[i]
    counts = data[col].value_counts().sort_index()
    ax.bar(counts.index.astype(str), counts.values, color='coral', edgecolor='white')
    ax.set_title(col, fontsize=10)
    ax.tick_params(labelsize=8)
for j in range(n, axes.size):
    axes.flat[j].set_visible(False)
plt.tight_layout()
plt.show()

# --- Target variable ---
fig, ax = plt.subplots(figsize=(5, 3))
counts = data[target_col].value_counts().sort_index()
ax.bar(['No Default (0)', 'Default (1)'], counts.values, color=['steelblue', 'coral'], edgecolor='white')
ax.set_title('Target Variable Distribution')
ax.set_ylabel('Count')
for i, v in enumerate(counts.values):
    ax.text(i, v + 200, f'{v} ({v/len(data)*100:.1f}%)', ha='center', fontsize=10)
plt.tight_layout()
plt.show()

## Dataset Preparation

### Data Cleaning

The data cleaning process is the procedure of correcting or removing incomplete, inaccurate, or incorrect portions of the dataset.

In [ ]:
data[['LIMIT_BAL','SEX', 'EDUCATION', 'MARRIAGE', 'AGE']].describe()

In [ ]:
summary = data['EDUCATION'].value_counts()
print(summary)

As far as EDUCATION is concerned there are three categories not listed in the description of the dataset provided by the UCI website that corresponds to 0, 5, and 6;

These rows are deleted

In [ ]:
m = (data['EDUCATION'] == 0)|(data['EDUCATION'] == 6)|(data['EDUCATION'] == 5)
data = data.drop(data.EDUCATION[m].index.values, axis=0)
summary = data['EDUCATION'].value_counts()
print(summary)

for MARRIAGE category from the function .describe() it is possible to notice that there is a minimum value equal to 0, that does not corresponds to any category previously described.

In [ ]:
summary = data['MARRIAGE'].value_counts()
print(summary)

In [ ]:
m = (data['MARRIAGE'] == 0)
data = data.drop(data.MARRIAGE[m].index.values, axis=0)

summary = data['MARRIAGE'].value_counts()
print(summary)

In [ ]:
data[['PAY_' + str(n) for n in [0, 2, 3, 4, 5, 6]]].describe()

As far as the attributes PAY_* is concerned, all of this attributes have a minimum value equal to -2, not included in the ranking. On the other hand the maximum value assumed is equal 8 so probably it is necessary a re-scaling of the attribute.

In [ ]:
data[['PAY_' + str(n) for n in [0, 2, 3, 4, 5, 6]]] += 1

### Preprocessing



#### One-hot Encoding for Categorical Variables

Categorical variables such as SEX, MARRIAGE, and EDUCATION are turned into one-hot vectors to remove any implicit ordering that has no meaning in this context.

In [ ]:
data['EDUCATION'] = data['EDUCATION'].astype('category')
data['SEX'] = data['SEX'].astype('category')
data['MARRIAGE'] = data['MARRIAGE'].astype('category')

data=pd.concat([pd.get_dummies(data['EDUCATION'], prefix='EDUCATION'),
                  pd.get_dummies(data['SEX'], prefix='SEX'),
                  pd.get_dummies(data['MARRIAGE'], prefix='MARRIAGE'),
                  data],axis=1)
data.drop(['EDUCATION'],axis=1, inplace=True)
data.drop(['SEX'],axis=1, inplace=True)
data.drop(['MARRIAGE'],axis=1, inplace=True)
data.head()

#### Min-Max Scaling

Input variables may have different units and therefore different scales. A MinMaxScaler is applied to scale the numerical features to the range $(0, 1)$.

$$X_{\text{std}} = \frac{X - X_{\min}}{X_{\max} - X_{\min}}$$

$$X_{\text{scaled}} = X_{\text{std}} \times (\text{max} - \text{min}) + \text{min}$$

where $(\text{min}, \text{max}) = (0, 1)$.

⚠️ **Important:** We must fit the scaler on the **training set only** and then apply (transform) it to the validation and test sets. Fitting the scaler on the entire dataset before splitting causes **data leakage** — information from the test set would influence the scaling parameters, giving an overly optimistic estimate of generalisation performance.

Scaling will be applied after the train/validation/test split — see below.

### Hold-out Validation

In hold-out validation we divide the data into 3 subsets:
1. **Training:** to learn the parameters (weights) of the model.
2. **Validation:** for tuning hyperparameters and model selection.
3. **Test:** to evaluate the final model's performance. **DO NOT** use this split to set or tune any element of the model.

For this example, let's divide the data into 60/20/20.

In [ ]:
from sklearn.model_selection import train_test_split

with pd.option_context('mode.chained_assignment', None):
    train_data_, test_data = train_test_split(data, test_size=0.2, shuffle=True,random_state=0)

with pd.option_context('mode.chained_assignment', None):
    train_data, val_data = train_test_split(train_data_, test_size=0.25, shuffle=True,random_state=0)

print(train_data.shape[0], val_data.shape[0], test_data.shape[0])

> ⚠️ **Important — The Golden Rule of Data Splitting**
>
> The test set must remain completely untouched during model development. Any use of test set performance to guide modelling choices (e.g., selecting hyperparameters, choosing architectures, or deciding when to stop training) **contaminates your evaluation** — the reported performance is no longer a reliable estimate of how the model will perform on truly unseen data.
>
> 🚨 **Course policy:** Peeking at the test set during the model development phase is a cardinal sin in this course. Doing so in your assignments will result in a **failing grade**. Train and tune on train/validation only; evaluate on test **exactly once** at the very end.

🤔 **Think:** Should we split the data *before* we start analysing it? Why or why not?

#### Min-Max Scaling

In [ ]:
from sklearn.preprocessing import MinMaxScaler

# Columns to scale (numerical features only — not the one-hot encoded ones)
num_cols = (
    ['LIMIT_BAL', 'AGE'] +
    ['BILL_AMT' + str(i) for i in range(1, 7)] +
    ['PAY_AMT' + str(i) for i in range(1, 7)] +
    ['PAY_' + str(i) for i in [0, 2, 3, 4, 5, 6]]
)

# Fit scaler on TRAINING data only, then transform all splits
scaler = MinMaxScaler()
train_data[num_cols] = scaler.fit_transform(train_data[num_cols])
val_data[num_cols] = scaler.transform(val_data[num_cols])
test_data[num_cols] = scaler.transform(test_data[num_cols])

print('Scaling complete — fitted on training data only.')

plot and see if the scaling is done correctly.

In [ ]:
# Separate column types in the processed data
binary_cols = [c for c in train_data.columns if train_data[c].nunique() <= 2 and c != 'default payment next month']
continuous_cols = [c for c in train_data.columns if c not in binary_cols and c != 'default payment next month']
target_col = 'default payment next month'

# --- Continuous features (after scaling) ---
n = len(continuous_cols)
ncols_plot = 4
nrows_plot = int(np.ceil(n / ncols_plot))

fig, axes = plt.subplots(nrows_plot, ncols_plot, figsize=(18, 3.5 * nrows_plot))
fig.suptitle('Distribution of Continuous Features (after scaling)', fontsize=14, y=1.01)
for i, col in enumerate(continuous_cols):
    ax = axes.flat[i]
    ax.hist(train_data[col], bins=30, alpha=0.5, color='steelblue', edgecolor='white', density=True)
    ax.set_title(col, fontsize=10)
    ax.tick_params(labelsize=8)
for j in range(n, axes.size):
    axes.flat[j].set_visible(False)
plt.tight_layout()
plt.show()

# --- Binary / one-hot features ---
n = len(binary_cols)
nrows_plot = int(np.ceil(n / ncols_plot))

fig, axes = plt.subplots(nrows_plot, ncols_plot, figsize=(18, 3.5 * nrows_plot))
fig.suptitle('Distribution of Binary / One-Hot Features', fontsize=14, y=1.01)
for i, col in enumerate(binary_cols):
    ax = axes.flat[i]
    counts = train_data[col].astype(float).value_counts().sort_index()
    ax.bar(counts.index.astype(str), counts.values, color='coral', edgecolor='white')
    ax.set_title(col, fontsize=10)
    ax.tick_params(labelsize=8)
for j in range(n, axes.size):
    axes.flat[j].set_visible(False)
plt.tight_layout()
plt.show()

# --- Target variable ---
fig, ax = plt.subplots(figsize=(5, 3))
counts = train_data[target_col].value_counts().sort_index()
ax.bar(['No Default (0)', 'Default (1)'], counts.values, color=['steelblue', 'coral'], edgecolor='white')
ax.set_title('Target Variable Distribution (Training Set)')
ax.set_ylabel('Count')
for i, v in enumerate(counts.values):
    ax.text(i, v + 100, f'{v} ({v/len(train_data)*100:.1f}%)', ha='center', fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
train_X = train_data.drop(['default payment next month'], axis=1).to_numpy().astype(np.float32)
train_y = train_data[['default payment next month']].to_numpy().astype(np.float32).ravel()

val_X = val_data.drop(['default payment next month'], axis=1).to_numpy().astype(np.float32)
val_y = val_data[['default payment next month']].to_numpy().astype(np.float32).ravel()

test_X = test_data.drop(['default payment next month'], axis=1).to_numpy().astype(np.float32)
test_y = test_data[['default payment next month']].to_numpy().astype(np.float32).ravel()


👉 **Note:** The scaler is fitted using `fit_transform()` on the training set, then applied using `transform()` (without refitting) on the validation and test sets. This ensures no information from val/test leaks into the preprocessing step.

The test set values may slightly exceed the $[0, 1]$ range — this is expected and correct, since the test set may contain values outside the range seen during training.

### Managing Dataset Imbalance

In machine learning, it is difficult to train an effective model if the class distribution is imbalanced. The overall accuracy may be high, but the minority class may be poorly classified. To tackle this problem, one can adopt two strategies:

- **Oversampling** the minority class
- **Undersampling** the majority class

Let's check the class balance:

In [ ]:
l0 = np.sum(train_y == 0)
l1 = np.sum(train_y == 1)

print(f'Class 0: {l0/(l0+l1)*100:.1f}%, Class 1: {l1/(l0+l1)*100:.1f}%')

#### SMOTE

The Synthetic Minority Oversampling Technique (SMOTE) generates synthetic samples by interpolating between existing minority class instances. This helps rebalance the training set without simply duplicating samples.

In [ ]:
from imblearn.over_sampling import SMOTE, KMeansSMOTE

def oversample_dataset(X_train, y_train):
    print(f'Before rebalancing: {len(X_train)} samples')
    oversample = SMOTE()
    X_resampled, y_resampled = oversample.fit_resample(X_train, y_train)

    l0 = np.sum(y_resampled == 0)
    l1 = np.sum(y_resampled == 1)
    print(f'After rebalancing: {len(X_resampled)} samples')
    print(f'  Class 0: {l0} ({l0/(l0+l1)*100:.1f}%), Class 1: {l1} ({l1/(l0+l1)*100:.1f}%)')
    return X_resampled, y_resampled

In [ ]:
X_train_balanced, y_train_balanced = oversample_dataset(train_X, train_y)

### Convert to PyTorch Tensors and DataLoaders

In [ ]:
# Convert to tensors
train_X_t = torch.tensor(X_train_balanced, dtype=torch.float32)
train_y_t = torch.tensor(y_train_balanced, dtype=torch.float32)

val_X_t = torch.tensor(val_X, dtype=torch.float32)
val_y_t = torch.tensor(val_y, dtype=torch.float32)

test_X_t = torch.tensor(test_X, dtype=torch.float32)
test_y_t = torch.tensor(test_y, dtype=torch.float32)

# Create DataLoader for mini-batch training
train_dataset = TensorDataset(train_X_t, train_y_t)
train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)

print(f'Training samples: {len(train_X_t)}')
print(f'Validation samples: {len(val_X_t)}')
print(f'Test samples: {len(test_X_t)}')

## Data Modelling

### Performance Measures

The metrics commonly used to evaluate a classifier are:

- **Accuracy:** the ratio of correct predictions (TP + TN) over the total number of predictions (TP + FN + FP + TN).
- **Precision** (positive predictive value): TP / (TP + FP) — how many of the predicted positives are actually positive.
- **Recall** (sensitivity): TP / (TP + FN) — how many of the actual positives were correctly identified.
- **F1-score:** the harmonic mean of precision and recall:

$$F_1 = 2 \cdot \frac{\text{precision} \cdot \text{recall}}{\text{precision} + \text{recall}}$$

In the case of class imbalance, accuracy alone can be misleading — a model that always predicts the majority class would still achieve high accuracy. Precision, recall, and F1-score provide a more complete picture, especially for the minority class.

We use sklearn's `classification_report()` to compute all of these.

**Terminology:**
- **TP** (true positive): correctly labelled as positive
- **TN** (true negative): correctly labelled as negative
- **FP** (false positive): incorrectly labelled as positive
- **FN** (false negative): incorrectly labelled as negative


Choosing a Single Target Metric  
👉 Important: While it is useful to monitor multiple metrics to get a holistic view of model behaviour, you must select a single metric as your target before starting model development. This is critical — without a clearly defined target metric, you risk going in circles trying to balance competing objectives and never knowing when to stop.

The target metric should reflect what matters most for your application. For example, in this credit default problem, a bank might care more about recall (catching as many actual defaulters as possible) than precision, or vice versa. The target metric could also be a weighted combination of several measures (e.g., a weighted F1-score), but it must be defined upfront as a single number.

Alongside the target metric, you should also identify a target value — the level of performance at which model development stops. This can be informed by business requirements, published benchmarks on similar tasks, or baseline model performance. Without a target value, there is no principled stopping criterion and development can drag on indefinitely.

🤔 Think: For this credit card default prediction task, which metric would you choose as the target, and what would be a reasonable target value? Consider the consequences of false positives (denying credit to a good customer) versus false negatives (approving credit for someone who will default).

### Define Hypothesis Space

We start with a simple feed-forward neural network: one hidden layer with 128 neurons and sigmoid activation, followed by a single output neuron for binary classification.

👉 **Note:** In PyTorch, we use `nn.BCEWithLogitsLoss` which combines a sigmoid activation with binary cross-entropy loss. This is more numerically stable than applying sigmoid separately and then computing the loss. Therefore, the output layer does **not** include a sigmoid activation.

In [ ]:
class SimpleNN(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.Sigmoid(),
            nn.Linear(128, 1),  # raw logits — BCEWithLogitsLoss applies sigmoid
        )

    def forward(self, x):
        return self.net(x)

input_dim = train_X_t.shape[1]
print(f'Input dimension: {input_dim}')

In [ ]:
train_X_t

In [ ]:
def train_model(model, train_loader, val_X, val_y,
                num_epochs=50, lr=0.001, model_name='model'):
    """
    Train a binary classification model and record history.

    Returns a dict with lists of train/val loss and metrics per epoch.
    """
    criterion = nn.BCEWithLogitsLoss()  # numerically stable binary cross-entropy
    optimizer = optim.Adam(model.parameters(), lr=lr)

    history = {
        'loss': [], 'val_loss': [],
        'accuracy': [], 'val_accuracy': [],
        'precision': [], 'val_precision': [],
        'recall': [], 'val_recall': [],
    }

    for epoch in range(num_epochs):
        # --- Training ---
        model.train()
        epoch_loss = 0.0
        all_preds, all_labels = [], []

        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)

            logits = model(X_batch).squeeze()
            loss = criterion(logits, y_batch)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item() * len(y_batch)
            preds = (torch.sigmoid(logits) > 0.5).float()
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y_batch.cpu().numpy())

        # Training metrics
        epoch_loss /= len(all_labels)
        all_preds, all_labels = np.array(all_preds), np.array(all_labels)
        history['loss'].append(epoch_loss)
        history['accuracy'].append(accuracy_score(all_labels, all_preds))
        history['precision'].append(precision_score(all_labels, all_preds, zero_division=0))
        history['recall'].append(recall_score(all_labels, all_preds, zero_division=0))

        # --- Validation ---
        model.eval()
        with torch.inference_mode():
            val_logits = model(val_X.to(device)).squeeze()
            val_loss = criterion(val_logits, val_y.to(device)).item()
            val_preds = (torch.sigmoid(val_logits) > 0.5).float().cpu().numpy()
            val_labels = val_y.cpu().numpy()

        history['val_loss'].append(val_loss)
        history['val_accuracy'].append(accuracy_score(val_labels, val_preds))
        history['val_precision'].append(precision_score(val_labels, val_preds, zero_division=0))
        history['val_recall'].append(recall_score(val_labels, val_preds, zero_division=0))

        if (epoch + 1) % max(1, num_epochs // 5) == 0:
            print(f'  Epoch {epoch+1}/{num_epochs} — '
                  f'Loss: {epoch_loss:.4f}, Acc: {history["accuracy"][-1]:.3f} | '
                  f'Val Loss: {val_loss:.4f}, Val Acc: {history["val_accuracy"][-1]:.3f}')

    return history

### Search for Optimal Hypothesis

In [ ]:
torch.manual_seed(0)
model1 = SimpleNN(input_dim).to(device)
print(model1)

m_histories['simple_1H'] = train_model(
    model1, train_loader, val_X_t, val_y_t,
    num_epochs=50, lr=0.001, model_name='simple_1H'
)

In [ ]:
plotter(m_histories, metric='accuracy', ylim=[0.0, 1.1])

In [ ]:
# Evaluate on VALIDATION set (not test!)
model1.eval()
with torch.inference_mode():
    val_logits = model1(val_X_t.to(device)).squeeze()
    val_preds = (torch.sigmoid(val_logits) > 0.5).float().cpu().numpy()

print('Validation Results:')
print(classification_report(val_y, val_preds))

🤔 **Think:** Why are we evaluating on the **validation** set here and not the test set?

We are still in the model development phase — we haven't finalised our model yet. Using the test set now to judge whether this model is good enough would contaminate our final evaluation. The test set should only be used **once**, at the very end, as a final certificate of generalisation performance.

🤔 **Think:** What should we do next to improve the model?

### A More Complex Model

Let's try a deeper network with:
- **ReLU** activations (better than sigmoid for hidden layers — avoids vanishing gradients)
- **Batch Normalisation** (stabilises and accelerates training)
- **Dropout** (regularisation to reduce overfitting)

In [ ]:
class ComplexNN(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Dropout(0.5),

            nn.Linear(128, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(0.5),

            nn.Linear(64, 32),
            nn.ReLU(),
            nn.BatchNorm1d(32),
            nn.Dropout(0.5),

            nn.Linear(32, 1),  # raw logits
        )

    def forward(self, x):
        return self.net(x)

torch.manual_seed(0)
model2 = ComplexNN(input_dim).to(device)
print(model2)

m_histories['complex_3H'] = train_model(
    model2, train_loader, val_X_t, val_y_t,
    num_epochs=100, lr=0.001, model_name='complex_3H'
)

In [ ]:
plotter(m_histories, metric='accuracy', ylim=[0.0, 1.1])

Evaluate model 2 on VALIDATION set

In [ ]:
model2.eval()
with torch.inference_mode():
    val_logits2 = model2(val_X_t.to(device)).squeeze()
    val_preds2 = (torch.sigmoid(val_logits2) > 0.5).float().cpu().numpy()

print('Validation Results (Model 2):')
print(classification_report(val_y, val_preds2))

### Final Evaluation on the Test Set

Now that we have finalised our model (model 2 — the deeper network with BatchNorm and Dropout), we evaluate it **once** on the held-out test set. This gives us an unbiased estimate of how the model would perform on truly unseen data.

🚨 **This is the only time we touch the test set.**

In [ ]:
# Final test evaluation — do this ONCE with the chosen model
model2.eval()
with torch.inference_mode():
    test_logits_final = model2(test_X_t.to(device)).squeeze()
    test_preds_final = (torch.sigmoid(test_logits_final) > 0.5).float().cpu().numpy()

print('='*60)
print('FINAL TEST SET EVALUATION')
print('='*60)
print(classification_report(test_y, test_preds_final))

## 📝 Exercises

1. Analyse the final model and see if you can explain the models behaviour?

2. Try different methods you know to improve the perfomance.